# Assignment 4 - Multimodal Interactions
**CS 5970/6970 | Spring 2026**

This notebook builds an agentic system that uses CLIP as a retrieval (RAG) system over images and BLIP for visual grounding via captioning.

## Stage 0: Environment setup

In [33]:
import os

REPO_URL = "https://github.com/dutch-casa/gen-ai-project-four.git"
REPO_DIR = "gen-ai-project-four"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    print(f"Cloned {REPO_URL}")
else:
    print(f"{REPO_DIR} already exists, skipping clone.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Cloning into 'gen-ai-project-four'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (114/114), done.
remote: Total 118 (delta 7), reused 112 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 11.99 MiB | 11.49 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Cloned https://github.com/dutch-casa/gen-ai-project-four.git
Working directory: /content/gen-ai-project-four/gen-ai-project-four/gen-ai-project-four


In [34]:
!pip install -q torch torchvision transformers ftfy regex Pillow

In [19]:
!pip install -q git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done


In [20]:
import os
import json
import random
import numpy as np
import torch
import clip
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

SEED = 3232004
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Seed: {SEED}")

Device: cuda
Seed: 3232004


## Stage 1: Dataset setup

In [21]:
IMAGE_DIR = "assignment4_images"

image_paths = sorted([
    os.path.join(IMAGE_DIR, f"image_{i:03d}.jpg")
    for i in range(100)
])

image_ids = [f"image_{i:03d}.jpg" for i in range(100)]

print(f"Loaded {len(image_paths)} image paths")
print(f"First: {image_paths[0]}")
print(f"Last:  {image_paths[-1]}")

missing = [p for p in image_paths if not os.path.exists(p)]
if missing:
    print(f"WARNING: {len(missing)} images missing!")
else:
    print("All 100 images found.")

Loaded 100 image paths
First: assignment4_images/image_000.jpg
Last:  assignment4_images/image_099.jpg
All 100 images found.


## Stage 2: CLIP as a RAG system

In [22]:
print("Loading CLIP model:")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
print("CLIP ViT-B/32 loaded.")

Loading CLIP model:
CLIP ViT-B/32 loaded.


In [23]:
print("Computing image embeddings:")

image_embeddings = []

with torch.no_grad():
    for i, path in enumerate(image_paths):
        img = clip_preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
        emb = clip_model.encode_image(img)
        image_embeddings.append(emb)
        if (i + 1) % 25 == 0:
            print(f"  {i + 1}/100 images embedded")

image_embeddings = torch.cat(image_embeddings, dim=0)
image_embeddings = image_embeddings / image_embeddings.norm(dim=-1, keepdim=True)

print(f"Embedding matrix shape: {image_embeddings.shape}")

Computing image embeddings:
  25/100 images embedded
  50/100 images embedded
  75/100 images embedded
  100/100 images embedded
Embedding matrix shape: torch.Size([100, 512])


In [24]:
def clip_retrieve(query: str, top_k: int = 5) -> list:
    text_tokens = clip.tokenize([query]).to(device)

    with torch.no_grad():
        text_emb = clip_model.encode_text(text_tokens)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

    sims = (image_embeddings @ text_emb.T).squeeze(1)
    top = sims.argsort(descending=True)[:top_k]

    return [(image_ids[i.item()], sims[i].item()) for i in top]

In [25]:
print("Stage 2 retrieval demo (5 queries, top-5 each):")

demo_queries = [
    "a photo of a dog",
    "a vehicle on a road",
    "people playing sports",
    "cooking food in a kitchen",
    "outdoor scene with people"
]

for query in demo_queries:
    results = clip_retrieve(query, top_k=5)
    print(f'\nQuery: "{query}"')
    for img_id, score in results:
        print(f"  {img_id}  score={score:.4f}")

Stage 2 retrieval demo (5 queries, top-5 each):

Query: "a photo of a dog"
  image_001.jpg  score=0.2646
  image_007.jpg  score=0.2593
  image_068.jpg  score=0.2505
  image_000.jpg  score=0.2441
  image_009.jpg  score=0.2440

Query: "a vehicle on a road"
  image_015.jpg  score=0.2379
  image_013.jpg  score=0.2341
  image_003.jpg  score=0.2216
  image_018.jpg  score=0.2192
  image_060.jpg  score=0.2178

Query: "people playing sports"
  image_054.jpg  score=0.2856
  image_011.jpg  score=0.2791
  image_026.jpg  score=0.2786
  image_027.jpg  score=0.2681
  image_020.jpg  score=0.2646

Query: "cooking food in a kitchen"
  image_078.jpg  score=0.2615
  image_032.jpg  score=0.2607
  image_035.jpg  score=0.2520
  image_030.jpg  score=0.2505
  image_033.jpg  score=0.2493

Query: "outdoor scene with people"
  image_034.jpg  score=0.2808
  image_049.jpg  score=0.2698
  image_011.jpg  score=0.2576
  image_004.jpg  score=0.2454
  image_016.jpg  score=0.2404


## Stage 3: Image captioning (BLIP)

In [26]:
print("Loading BLIP model (half-precision):")

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch.float16
).to(device)

print("BLIP loaded in float16.")

Loading BLIP model (half-precision):


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

BLIP loaded in float16.


In [27]:
def caption_image(image_path: str) -> str:
    img = Image.open(image_path).convert("RGB")
    inputs = blip_processor(images=img, return_tensors="pt").to(device, torch.float16)

    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=50)

    return blip_processor.decode(out[0], skip_special_tokens=True)

In [28]:
print("Stage 3 captioning demo (5 images):")

demo_image_indices = [0, 20, 40, 60, 80]
for idx in demo_image_indices:
    path = image_paths[idx]
    cap = caption_image(path)
    print(f"  {image_ids[idx]}: {cap}")

Stage 3 captioning demo (5 images):
  image_000.jpg: a dog sitting on a red couch with a book
  image_020.jpg: a baseball player is getting ready to hit the ball
  image_040.jpg: a young child playing with toys in a room
  image_060.jpg: a horse walking through a dirt covered field
  image_080.jpg: a zebra standing in a dirt field next to a fence


## Stage 4: Agentic RAG pipeline

In [29]:
def agentic_query(query: str, top_k: int = 5, caption_top: int = 3) -> dict:
    tool_trace = []

    # If the query names a specific file, skip retrieval and caption it.
    direct = next((img_id for img_id in image_ids if img_id in query), None)

    if direct:
        path = os.path.join(IMAGE_DIR, direct)
        cap = caption_image(path)
        tool_trace.append(f"caption_image({direct})")
        return {
            "query": query,
            "retrieved_images": [direct],
            "selected_image": direct,
            "caption": cap,
            "tool_trace": tool_trace,
            "final_answer": f"{direct}: {cap}"
        }

    retrieved = clip_retrieve(query, top_k=top_k)
    tool_trace.append(f"clip_retrieve('{query}', top_k={top_k})")
    retrieved_ids = [img_id for img_id, _ in retrieved]

    captions = {}
    for img_id, _ in retrieved[:caption_top]:
        captions[img_id] = caption_image(os.path.join(IMAGE_DIR, img_id))
        tool_trace.append(f"caption_image({img_id})")

    # Re-rank: whichever caption lands closest to the query in CLIP text space wins.
    with torch.no_grad():
        q_emb = clip_model.encode_text(clip.tokenize([query]).to(device))
        q_emb = q_emb / q_emb.norm(dim=-1, keepdim=True)

        best_image, best_score = retrieved_ids[0], -1.0
        for img_id, cap in captions.items():
            c_emb = clip_model.encode_text(clip.tokenize([cap]).to(device))
            c_emb = c_emb / c_emb.norm(dim=-1, keepdim=True)
            score = (q_emb @ c_emb.T).item()
            if score > best_score:
                best_image, best_score = img_id, score

    best_caption = captions.get(best_image, "")

    return {
        "query": query,
        "retrieved_images": retrieved_ids,
        "selected_image": best_image,
        "caption": best_caption,
        "tool_trace": tool_trace,
        "final_answer": (
            f"Retrieved {len(retrieved_ids)} candidates. "
            f"Best match: {best_image}. Caption: {best_caption}"
        )
    }

## Stage 5: Query evaluation (15 queries)

In [30]:
queries = [
    # A. Retrieval queries
    "Find me a picture of a dog.",
    "Find me a picture of a vehicle.",
    "Find me an image showing sports.",
    "Find me an image that likely involves cooking.",
    "Find me an outdoor scene with people.",
    # B. Direct image inspection
    "Here is the path to image_005.jpg. Describe it.",
    "Here is the path to image_021.jpg. What is happening?",
    "Describe image_044.jpg in one sentence.",
    "What objects are visible in image_073.jpg?",
    "Describe image_090.jpg briefly.",
    # C. Retrieval + grounding
    "Find me a picture of people eating and describe it.",
    "Find me an image of a person riding something and summarize it.",
    "Find an image that appears to show a kitchen scene and explain why.",
    "Find an image with animals outdoors and describe the scene.",
    'Find the best image for "family meal" and describe it.'
]

print(f"Running {len(queries)} queries:")

Running 15 queries:


In [31]:
all_results = []

for i, q in enumerate(queries, 1):
    print(f"\nQuery {i}: {q}")
    result = agentic_query(q)
    all_results.append(result)
    print(f"  Retrieved: {result['retrieved_images']}")
    print(f"  Selected:  {result['selected_image']}")
    print(f"  Caption:   {result['caption']}")
    print(f"  Tools:     {result['tool_trace']}")
    print(f"  Answer:    {result['final_answer']}")


Query 1: Find me a picture of a dog.
  Retrieved: ['image_001.jpg', 'image_000.jpg', 'image_007.jpg', 'image_068.jpg', 'image_006.jpg']
  Selected:  image_007.jpg
  Caption:   a dog is standing in a living room
  Tools:     ["clip_retrieve('Find me a picture of a dog.', top_k=5)", 'caption_image(image_001.jpg)', 'caption_image(image_000.jpg)', 'caption_image(image_007.jpg)']
  Answer:    Retrieved 5 candidates. Best match: image_007.jpg. Caption: a dog is standing in a living room

Query 2: Find me a picture of a vehicle.
  Retrieved: ['image_013.jpg', 'image_015.jpg', 'image_040.jpg', 'image_019.jpg', 'image_095.jpg']
  Selected:  image_015.jpg
  Caption:   a car parked in front of a white bus
  Tools:     ["clip_retrieve('Find me a picture of a vehicle.', top_k=5)", 'caption_image(image_013.jpg)', 'caption_image(image_015.jpg)', 'caption_image(image_040.jpg)']
  Answer:    Retrieved 5 candidates. Best match: image_015.jpg. Caption: a car parked in front of a white bus

Query 3: Find

In [32]:
print("Full JSON output:")
print(json.dumps(all_results, indent=2))

Full JSON output:
[
  {
    "query": "Find me a picture of a dog.",
    "retrieved_images": [
      "image_001.jpg",
      "image_000.jpg",
      "image_007.jpg",
      "image_068.jpg",
      "image_006.jpg"
    ],
    "selected_image": "image_007.jpg",
    "caption": "a dog is standing in a living room",
    "tool_trace": [
      "clip_retrieve('Find me a picture of a dog.', top_k=5)",
      "caption_image(image_001.jpg)",
      "caption_image(image_000.jpg)",
      "caption_image(image_007.jpg)"
    ],
    "final_answer": "Retrieved 5 candidates. Best match: image_007.jpg. Caption: a dog is standing in a living room"
  },
  {
    "query": "Find me a picture of a vehicle.",
    "retrieved_images": [
      "image_013.jpg",
      "image_015.jpg",
      "image_040.jpg",
      "image_019.jpg",
      "image_095.jpg"
    ],
    "selected_image": "image_015.jpg",
    "caption": "a car parked in front of a white bus",
    "tool_trace": [
      "clip_retrieve('Find me a picture of a vehicle.',

## Stage 6: Insight blocks

### Block 1: CLIP as RAG

CLIP was solid for concrete, single-object queries. "A photo of a dog" pulled image_001, image_007, and image_000 to the top, and BLIP confirmed each one was actually a dog. "A vehicle on a road" surfaced image_013 and image_015; the selected result captioned as "a car parked in front of a white bus." Scores were spread enough (roughly 0.21 to 0.26) that the ordering wasn't noise.

Situational queries were rougher. For "cooking food in a kitchen," CLIP put image_035 on top, but the re-rank step bumped image_078 to selected once BLIP read it as "a man in a kitchen preparing food." Right neighborhood, wrong top-1. "Outdoor scene with people" had a similar story: CLIP led with image_004, the re-ranker moved image_054 up, and then BLIP captioned that one as "a man is throwing a frurd frck." Gibberish. CLIP got the semantics close, but nothing downstream noticed the caption was broken.

### Block 2: Retrieval vs captioning

Captioning was redundant for queries 1-3 (dog, vehicle, sports). CLIP's top hits already matched the intent; BLIP just confirmed what the scores said. Query 3 is a minor exception: CLIP ranked image_026 first, but re-ranking against BLIP's caption for image_028 ("a baseball player is getting ready to hit the ball") won out. Both were sports images, so the win was picking a better representative, not a better category.

Captioning actually mattered on queries 4, 12, and 13. Query 4 and 13 both concerned kitchens: CLIP's top-1 wasn't the best fit either time, and re-ranking pushed image_078 forward because its caption literally said "a man in a kitchen preparing food." Query 12 went the same way: CLIP put image_022 first, BLIP's caption on image_087 ("a person riding a horse across a field") answered the query more directly, and re-ranking made the swap.

### Block 3: Agentic behavior

Routing worked. Queries 6-10 name a filename, so the pipeline skipped CLIP and called only `caption_image` — the tool traces show a single call each. Retrieval queries ran `clip_retrieve` once, then `caption_image` on three candidates: four calls total.

Good tool use: query 13 ("kitchen scene"). CLIP led with image_039, but after captioning the top three, re-ranking picked image_078 ("a man in a kitchen preparing food"). The caption was a direct answer to the query. Retrieval narrowed the field, captioning chose the winner.

Wasteful tool use: query 1 ("picture of a dog"). All five retrieved images were plausibly dogs. The pipeline captioned three of them and swapped image_001 for image_007, which was also a dog. Three BLIP calls to make a lateral move. For queries this concrete, top-1 from CLIP would have been fine and captioning was overhead.

### Block 4: Failure case

Query 14 ("Find an image with animals outdoors and describe the scene") is the obvious failure. CLIP retrieved image_051, image_060, and image_082. BLIP then captioned image_082 as "a gi gi gi gi gi gi..." looping to the token limit. BLIP's decoder falls into repetition on certain images under float16, and this was one of them.

The break is on captioning, not retrieval — CLIP's three candidates were all reasonable. Where the pipeline failed was the re-rank step: it scored the repetition caption highest against the query embedding, probably because "gi" tokens produced a weird but high-similarity text embedding. A repetition guard (reject captions where any token appears more than N times) would have caught this and fallen back to the next candidate. Query 5 has the same shape: "a frurd frck" is nonsense, still won re-ranking, still surfaced as the final caption.

### Block 5: Grounding vs hallucination

Every final answer traces back to either a CLIP score or a BLIP caption. Nothing in the pipeline invents content. Direct-inspection answers (queries 6-10) are the raw caption. Retrieval answers cite the selected image and its caption.

The system does inherit BLIP's failures, though. The "gi gi gi" output in query 14 and "frurd frck" in query 5 aren't hallucinations in the fabricated-fact sense; they're decoder breakdowns. But the downstream effect is the same — the final answer contains text that doesn't describe the image.

There's also a softer grounding issue on query 11 ("people eating"). The selected image was image_031 with the caption "a group of people sitting around a table." People + table isn't the same as people eating, but the pipeline treated the retrieval match plus the tangentially related caption as confirmation. The answer is visually plausible but not strictly supported by what BLIP actually said.